# Notebook 05: Hand-Check of Claude's Labels

**Team:** Teenna Pachika and Johanna Garcia

Before we build on Claude's labels, we check them by hand. We pick 30 random negative reviews that Claude labeled, read each one, and mark whether Claude's labels are right.

In [3]:
# Set up
import sys
sys.path.append("..")

import os
import pandas as pd
from src.preprocessing import load_reviews

LABELS_FILE = "../data/claude_labels.csv"
CHECK_FILE = "../data/hand_check.csv"       # the file we fill in by hand

# The labels file only has the review ID and Claude's labels, so we add the review text from the main data
labels = pd.read_csv(LABELS_FILE)
reviews = load_reviews()[["reviewId", "content"]]
labels = labels.merge(reviews, on="reviewId")

print(f"Reviews labeled by Claude: {len(labels):,}")
labels.head()

Reviews labeled by Claude: 4,147


,reviewId,claude_labels,content
0,c8e7bf13-cd51-4412-b382-8a82ae84f394,"[""icon_notifications""]",Duolingo is an Excellent app. The main problem...
1,8fae69be-40e5-4feb-8fd7-9cc996071036,"[""ads"", ""paywall""]",Genuinely horrible app. The learning is very w...
2,7c83cd69-b946-439a-b472-21cfc56d38df,"[""ads"", ""paywall"", ""teaching_quality""]",It started off as fun learning. Then come the ...
3,39c8d5af-385c-4976-89f6-11d8d831d366,"[""teaching_quality""]","vocabulary, not sentences. duolingo doesn't wo..."
4,e4d8061d-6d77-4d51-8fee-14b4e36b8025,"[""energy_hearts""]",its not fair. the quest are to hard for me to ...


## 1. Create the Hand-Check File
The next cell saves 30 random reviews to `data/hand_check.csv`, with Claude's labels and three empty columns to fill in:
- `claude_correct`: **y** (all labels right), **partial** (some right), or **n** (none right)
- `claude_missed`: complaint types Claude should have added, or blank
- `notes`: anything worth reporting, like a clear Claude mistake (useful for the AI Usage Log)

Open the file in Excel or VS Code, fill it in, and save it as CSV.

In [4]:
if not os.path.exists(CHECK_FILE):   # don't overwrite our answers on a re-run
    check = labels.sample(30, random_state=4360)[["reviewId", "content", "claude_labels"]].copy()
    for col in ["claude_correct", "claude_missed", "notes"]:
        check[col] = ""
    check.to_csv(CHECK_FILE, index=False)
    print("Wrote hand_check.csv: fill it in, save it, then run the next cell")
else:
    print("hand_check.csv already exists: our answers are kept")

Wrote hand_check.csv: fill it in, save it, then run the next cell


## 2. Results

In [ ]:
checked = pd.read_csv(CHECK_FILE).fillna("")
answers = checked["claude_correct"].str.strip().str.lower()

print("Answers:", answers.value_counts().to_dict())
print(f"Claude fully correct: {(answers == 'y').sum()} of {len(checked)} ({(answers == 'y').mean():.0%})")
print(f"Fully or partly correct: {answers.isin(['y', 'partial']).sum()} of {len(checked)}")

missed = [t.strip() for cell in checked["claude_missed"] for t in str(cell).split(",") if t.strip()]
print("Complaints Claude missed most:", pd.Series(missed).value_counts().head(5).to_dict() if missed else "none")

In [ ]:
# Reviews where Claude was wrong or only partly right
pd.set_option("display.max_colwidth", None)
checked[answers.isin(["n", "partial"])][["content", "claude_labels", "claude_missed", "notes"]]

**What we found:** *(fill in after the check: how often Claude was right, which complaint it missed most, and one example of a mistake)*